# Quickstart: Postgres → BigQuery via Serverless Spark

This notebook shows the minimal path from a Cloud SQL Postgres table to a BigQuery table using Dataproc Serverless Spark — no infrastructure code, no config files, no orchestration tooling.

**What happens:**
1. Upload the PostgreSQL JDBC driver JAR to GCS (one-time).
2. Spin up a Dataproc Serverless Spark session from this notebook.
3. Read a Postgres table via JDBC (JDBC URL pulled from Secret Manager).
4. Write the result to BigQuery as a `CREATE OR REPLACE TABLE`.
5. Verify the result.

**Prerequisites:** Run `make infra-up && make seed` from the repo root first. This provisions Cloud SQL, Secret Manager, the GCS bucket, BQ datasets, and the service account, and seeds the database with sample data.

## Cell 1 — Configuration

Edit these variables to match your environment. Everything else in the notebook derives from them.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
PROJECT        = "johanesa-playground-326616"
REGION         = "us-central1"
GCS_BUCKET     = "johanesa-playground-326616-spark-etl-demo"

# Secret Manager resource name for the Postgres JDBC URL.
# Format: projects/<project>/secrets/<secret>/versions/latest
SECRET_NAME    = f"projects/{PROJECT}/secrets/thelook-db-jdbc-url/versions/latest"

# Service account created by infra/setup.sh.
# Must have roles: dataproc.worker, storage.objectAdmin,
#                  bigquery.dataEditor, bigquery.jobUser,
#                  secretmanager.secretAccessor
SERVICE_ACCOUNT = f"spark-etl-sa@{PROJECT}.iam.gserviceaccount.com"

# Postgres table to read (schema.table)
POSTGRES_TABLE = "public.orders"

# BigQuery destination
BQ_DATASET     = "raw_thelook"
BQ_TABLE       = "orders_quickstart"
# ──────────────────────────────────────────────────────────────────────────────

JAR_VERSION    = "42.7.3"
JAR_FILENAME   = f"postgresql-{JAR_VERSION}.jar"
JAR_GCS_PATH   = f"gs://{GCS_BUCKET}/jars/{JAR_FILENAME}"
BQ_TABLE_FQN   = f"{PROJECT}.{BQ_DATASET}.{BQ_TABLE}"

print(f"Project  : {PROJECT}")
print(f"Region   : {REGION}")
print(f"Source   : postgres/{POSTGRES_TABLE}")
print(f"Target   : {BQ_TABLE_FQN}")
print(f"JDBC JAR : {JAR_GCS_PATH}")

## Cell 2 — Upload JDBC driver to GCS (one-time)

Spark executors run remotely on Dataproc Serverless and need the PostgreSQL JDBC JAR on their classpath. We download it from the official source and upload it to GCS once. **Skip this cell if the JAR already exists** (e.g. you ran `make deploy`).

In [ ]:
import subprocess
from google.cloud import storage

JAR_URL = f"https://jdbc.postgresql.org/download/{JAR_FILENAME}"
JAR_GCS_PREFIX = f"jars/{JAR_FILENAME}"

# Check if JAR already exists in GCS to avoid unnecessary download.
gcs_client = storage.Client(project=PROJECT)
bucket = gcs_client.bucket(GCS_BUCKET)
blob = bucket.blob(JAR_GCS_PREFIX)

if blob.exists():
    print(f"JAR already in GCS: {JAR_GCS_PATH} — skipping download.")
else:
    print(f"Downloading {JAR_FILENAME} from jdbc.postgresql.org ...")
    result = subprocess.run(
        ["wget", "-q", "-O", f"/tmp/{JAR_FILENAME}", JAR_URL],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"wget failed: {result.stderr}")
    print(f"Downloaded to /tmp/{JAR_FILENAME}")

    print(f"Uploading to {JAR_GCS_PATH} ...")
    blob.upload_from_filename(f"/tmp/{JAR_FILENAME}")
    print("Upload complete.")

## Cell 3 — Create Spark session

This spins up a Dataproc Serverless Spark cluster via [Spark Connect](https://spark.apache.org/docs/latest/spark-connect-overview.html). The cluster is fully managed — no VMs to provision. Startup takes ~60–90 seconds.

Key session settings:
- `spark.jars` — makes the JDBC JAR available on the driver and executor classpaths.
- `service_account` — the SA that Spark executors run as (needs BQ + Secret Manager + Cloud SQL access).

In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from google.cloud.dataproc_v1 import Session

session = Session()

# Make the PostgreSQL JDBC JAR available on all Spark nodes.
session.runtime_config.properties["spark.jars"] = JAR_GCS_PATH

# Run the Spark session as our dedicated service account.
session.environment_config.execution_config.service_account = SERVICE_ACCOUNT

spark = (
    DataprocSparkSession.builder
    .appName("postgres-to-bq-quickstart")
    .dataprocSessionConfig(session)
    .getOrCreate()
)

print(f"Spark version : {spark.version}")
print("Session ready.")

## Cell 4 — Read JDBC URL from Secret Manager and read Postgres table

The JDBC URL (including host and credentials) is stored in Secret Manager — never hardcoded. We fetch it once from the notebook runtime (which uses your user credentials), then pass it to Spark.

In [ ]:
from google.cloud import secretmanager

# Fetch JDBC URL from Secret Manager (runs on the notebook runtime, not on Spark).
sm_client = secretmanager.SecretManagerServiceClient()
response = sm_client.access_secret_version(name=SECRET_NAME)
jdbc_url = response.payload.data.decode("utf-8").strip()
print(f"JDBC URL retrieved (host redacted): jdbc:postgresql://<host>/...")

# Read the Postgres table via JDBC.
# The URL already contains user/password so we only need the driver class.
df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", POSTGRES_TABLE)
    .option("driver", "org.postgresql.Driver")
    .load()
)

print(f"Table '{POSTGRES_TABLE}' loaded into Spark DataFrame.")

## Cell 5 — Preview the data

In [ ]:
df.printSchema()
df.show(10, truncate=False)
print(f"Total rows: {df.count()}")

## Cell 6 — Write to BigQuery

BQ Studio notebooks pre-configure the Spark BigQuery connector to use the DIRECT write method (Storage Write API). `mode("overwrite")` creates or replaces the table — safe to re-run.

In [ ]:
(
    df.write
    .format("bigquery")
    .option("table", BQ_TABLE_FQN)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)

print(f"Written to BigQuery: {BQ_TABLE_FQN}")

## Cell 7 — Verify in BigQuery

Read the table back from BigQuery to confirm the write succeeded.

In [ ]:
bq_df = (
    spark.read
    .format("bigquery")
    .option("table", BQ_TABLE_FQN)
    .load()
)

print(f"Rows in BigQuery: {bq_df.count()}")
bq_df.show(5, truncate=False)

## Cell 8 — Stop the Spark session

Dataproc Serverless sessions are billed while active. Stop explicitly when done, or let the session idle-timeout (default: 1 hour).

In [ ]:
spark.stop()
print("Spark session stopped.")